# 01 - HydroServer Things and Metadata

## Setup and Creation Controls
Configure the workspace, authentication, and safe cleanup behavior before creating anything.

In [ ]:
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path

import pandas as pd

try:
    from hydroserverpy import HydroServer
except Exception as exc:
    HydroServer = None
    print(f"hydroserverpy is not available yet: {exc}")

HYDROSERVER_HOST = "https://playground.hydroserver.org"
WORKSPACE_NAME = "hydroserver_uganda_demo"
WORKSPACE_IS_PRIVATE = False

# Choose one: "anonymous" or "api_key".
AUTH_METHOD = "anonymous"
HYDROSERVER_API_KEY = ""  # Keep secrets out of saved notebooks. Paste only when prompted.

# Facilitator controls. Defaults keep notebooks safe for anonymous/local runs.
CREATE_WORKSPACE_IF_MISSING = False
DELETE_CREATED_RESOURCES_AT_END = False
DEMO_RESOURCE_PREFIX = "Uganda Demo"
DEMO_RUN_SUFFIX = ""

# Google Colab/local path support. Leave blank unless data is somewhere custom.
DATA_DIR_OVERRIDE = ""


def resolve_data_dir(data_dir_override=""):
    candidates = []
    if data_dir_override:
        candidates.append(Path(data_dir_override).expanduser())
    candidates.extend([
        Path("data"),
        Path("../data"),
        Path("hydroserver_workshop/data"),
        Path("../hydroserver_workshop/data"),
        Path("/content/hydroserver_workshop/data"),
        Path("/content/data"),
    ])
    for candidate in candidates:
        if candidate.exists() and ((candidate / "Uganda_Hydroweb.csv").exists() or (candidate / "subset" / "stations.csv").exists()):
            return candidate
    searched = "\n".join(f"- {candidate}" for candidate in candidates)
    raise FileNotFoundError(
        "Could not find the workshop data folder. Upload hydroserver_workshop/data, "
        "upload data/ beside the notebook, or set DATA_DIR_OVERRIDE. Searched:\n"
        f"{searched}"
    )


DATA_DIR = resolve_data_dir(DATA_DIR_OVERRIDE)
USE_DATA_SUBSET = True
SUBSET_DATA_DIR = DATA_DIR / "subset"
USING_DATA_SUBSET = USE_DATA_SUBSET and SUBSET_DATA_DIR.exists()

STATION_CATALOG_CSV = SUBSET_DATA_DIR / "stations.csv" if USING_DATA_SUBSET else DATA_DIR / "Uganda_Hydroweb.csv"
SELECTED_STATION_CSV = SUBSET_DATA_DIR / "uganda_selected_station.csv" if USING_DATA_SUBSET and (SUBSET_DATA_DIR / "uganda_selected_station.csv").exists() else DATA_DIR / "uganda_selected_station.csv"
HYDROWEB_DIR = SUBSET_DATA_DIR / "hydroweb" if USING_DATA_SUBSET else DATA_DIR / "hydroweb"
GEOGLOWS_DIR = SUBSET_DATA_DIR / "geoglows" if USING_DATA_SUBSET else DATA_DIR / "geoglows"
STREAMFLOW_CSV = DATA_DIR / "sample_streamflow_observations.csv"
FORECAST_CSV = DATA_DIR / "sample_forecast_timeseries.csv"

demo_run_suffix = DEMO_RUN_SUFFIX or datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
demo_resource_prefix = f"{DEMO_RESOURCE_PREFIX} {demo_run_suffix}"

print(f"HydroServer host: {HYDROSERVER_HOST}")
print(f"Workspace: {WORKSPACE_NAME}")
print(f"Authentication mode: {AUTH_METHOD}")
print(f"Data directory: {DATA_DIR}")
print(f"Using subset data: {USING_DATA_SUBSET}")

## Connect to HydroServer

In [ ]:
def _prompt_if_needed(value, prompt):
    return value if value else getpass(prompt)

hs_api = None

if HydroServer is None:
    print("Install hydroserverpy before connecting to HydroServer.")
elif AUTH_METHOD == "anonymous":
    try:
        hs_api = HydroServer(host=HYDROSERVER_HOST)
        print("Connected anonymously. Anonymous mode can read public data but cannot create or upload resources.")
    except Exception as exc:
        print(f"Anonymous connection failed: {exc}")
elif AUTH_METHOD == "api_key":
    api_key = _prompt_if_needed(HYDROSERVER_API_KEY, "HydroServer API key: ")
    try:
        hs_api = HydroServer(host=HYDROSERVER_HOST, apikey=api_key)
        print("Connected with API-key authentication.")
    except Exception as exc:
        print(f"API-key connection failed: {exc}")
else:
    raise ValueError("AUTH_METHOD must be 'anonymous' or 'api_key'.")

## Track Created Resources

In [ ]:
created_resources = []


def resource_uid(resource):
    if resource is None:
        return None
    if isinstance(resource, str):
        return resource
    if isinstance(resource, dict):
        for key in ("uid", "id", "workspace_id"):
            if resource.get(key):
                return str(resource[key])
        return None
    for attr in ("uid", "id", "workspace_id"):
        value = getattr(resource, attr, None)
        if value:
            return str(value)
    if hasattr(resource, "model_dump"):
        dumped = resource.model_dump()
        if isinstance(dumped, dict):
            return resource_uid(dumped)
    return None


def resource_name(resource):
    if resource is None:
        return None
    if isinstance(resource, dict):
        for key in ("name", "code", "definition", "sampling_feature_code"):
            if resource.get(key):
                return str(resource[key])
        return None
    for attr in ("name", "code", "definition", "sampling_feature_code"):
        value = getattr(resource, attr, None)
        if value:
            return str(value)
    if hasattr(resource, "model_dump"):
        dumped = resource.model_dump()
        if isinstance(dumped, dict):
            return resource_name(dumped)
    return type(resource).__name__


def record_resource(resource_type, resource, station_id=None, source=None):
    created_resources.append({
        "resource_type": resource_type,
        "station_id": station_id,
        "source": source,
        "name_or_code": resource_name(resource),
        "uuid": resource_uid(resource),
        "python_type": type(resource).__name__,
        "resource": resource,
    })
    print(f"{resource_type}: {resource_name(resource)} | uuid={resource_uid(resource)}")
    return resource


def created_resources_dataframe(include_objects=False):
    rows = []
    for row in created_resources:
        rows.append({key: value for key, value in row.items() if include_objects or key != "resource"})
    return pd.DataFrame(rows)

print("Resource registry initialized.")

## Find or Optionally Create Demo Workspace

In [ ]:
workspace = None
workspace_uid = None

if hs_api is None:
    print("Skipping workspace lookup because the HydroServer client is unavailable.")
elif AUTH_METHOD == "anonymous":
    print(f"Anonymous mode: cannot create or manage workspace '{WORKSPACE_NAME}'.")
    print("Switch AUTH_METHOD to 'api_key' for live creation/upload demos.")
else:
    try:
        workspaces = hs_api.workspaces.list(fetch_all=True)
        workspace_items = getattr(workspaces, "items", workspaces)
        workspace = next((item for item in workspace_items if getattr(item, "name", None) == WORKSPACE_NAME), None)
        if workspace is None and CREATE_WORKSPACE_IF_MISSING:
            workspace = record_resource(
                "workspace",
                hs_api.workspaces.create(name=WORKSPACE_NAME, is_private=WORKSPACE_IS_PRIVATE),
            )
        elif workspace is None:
            print(f"Workspace '{WORKSPACE_NAME}' was not found. Ask the facilitator to create it first.")
        else:
            print(f"Using existing workspace: {getattr(workspace, 'name', WORKSPACE_NAME)}")
        workspace_uid = resource_uid(workspace)
        if workspace_uid:
            print(f"Workspace UUID: {workspace_uid}")
    except Exception as exc:
        print(f"Could not find or create workspace '{WORKSPACE_NAME}': {exc}")

## Prepare Thing and Metadata Templates

In [ ]:
selected_station = pd.read_csv(SELECTED_STATION_CSV).iloc[0]


def optional_float(value, default=None):
    if pd.isna(value):
        return default
    return float(value)


def build_thing_template(station):
    return {
        "name": f"{demo_resource_prefix} {station['ID']} {station.get('Name', 'Uganda Station')}",
        "description": "Uganda Hydroweb/GEOGLOWS station used for workshop observations.",
        "sampling_feature_type": "Site",
        "sampling_feature_code": str(station["ID"]),
        "site_type": "Stream",
        "latitude": optional_float(station["Latitude"]),
        "longitude": optional_float(station["Longitude"]),
        "elevation_m": optional_float(station.get("Elevation"), 0.0),
        "elevation_datum": str(station.get("Ellipsoid", "WGS84")),
        "state": "",
        "county": str(station.get("River", "")),
        "country": "UG",
        "data_disclaimer": "Workshop demonstration data from local Hydroweb and GEOGLOWS CSV files.",
        "is_private": False,
        "workspace": workspace_uid or "<workspace-uuid>",
    }

water_level_observed_property_template = {
    "name": f"{demo_resource_prefix} Water Level",
    "definition": "Satellite altimetry water level",
    "description": "Hydroweb/Theia water level for Uganda stations.",
    "observed_property_type": "Hydrology",
    "code": f"WaterLevel_{demo_run_suffix}",
    "workspace": workspace_uid or "<workspace-uuid>",
}
streamflow_observed_property_template = {
    "name": f"{demo_resource_prefix} Streamflow",
    "definition": "Water discharge in a river channel",
    "description": "GEOGLOWS streamflow for Uganda stations.",
    "observed_property_type": "Hydrology",
    "code": f"Streamflow_{demo_run_suffix}",
    "workspace": workspace_uid or "<workspace-uuid>",
}
water_level_unit_template = {
    "name": f"{demo_resource_prefix} Meter",
    "symbol": "m",
    "definition": "Meter",
    "unit_type": "Length",
    "workspace": workspace_uid or "<workspace-uuid>",
}
streamflow_unit_template = {
    "name": f"{demo_resource_prefix} Cubic meters per second",
    "symbol": "m3/s",
    "definition": "Cubic meters per second",
    "unit_type": "Discharge",
    "workspace": workspace_uid or "<workspace-uuid>",
}
hydroweb_sensor_template = {
    "name": f"{demo_resource_prefix} Hydroweb Theia Altimetry",
    "description": "Hydroweb/Theia satellite altimetry water-level source.",
    "encoding_type": "application/json",
    "manufacturer": "Theia Hydroweb",
    "sensor_model": str(selected_station.get("Missions", "Hydroweb")),
    "sensor_model_link": "https://catalogue.theia.data-terra.org/collection/HYDROWEB_RIVERS_OPE",
    "method_type": "Satellite altimetry",
    "method_link": "https://catalogue.theia.data-terra.org/collection/HYDROWEB_RIVERS_OPE",
    "method_code": f"HYDROWEB_{demo_run_suffix}",
    "workspace": workspace_uid or "<workspace-uuid>",
}
geoglows_sensor_template = {
    "name": f"{demo_resource_prefix} GEOGLOWS RFS",
    "description": "GEOGLOWS modeled streamflow source.",
    "encoding_type": "application/json",
    "manufacturer": "GEOGLOWS",
    "sensor_model": "GEOGLOWS RFS",
    "sensor_model_link": "https://data.geoglows.org/",
    "method_type": "Model",
    "method_link": "https://data.geoglows.org/",
    "method_code": f"GEOGLOWS_{demo_run_suffix}",
    "workspace": workspace_uid or "<workspace-uuid>",
}
processing_level_template = {
    "code": f"RAW_{demo_run_suffix}",
    "definition": "Raw",
    "explanation": "Data have not been processed or quality controlled in the workshop.",
    "workspace": workspace_uid or "<workspace-uuid>",
}
result_qualifier_template = {
    "code": f"SUSPECT_{demo_run_suffix}",
    "description": "Observation should be reviewed before operational use.",
    "workspace": workspace_uid or "<workspace-uuid>",
}

metadata_templates = pd.DataFrame([
    {"resource_type": "thing", "name_or_code": build_thing_template(selected_station)["name"]},
    {"resource_type": "observed_property", "name_or_code": water_level_observed_property_template["code"]},
    {"resource_type": "observed_property", "name_or_code": streamflow_observed_property_template["code"]},
    {"resource_type": "unit", "name_or_code": water_level_unit_template["symbol"]},
    {"resource_type": "unit", "name_or_code": streamflow_unit_template["symbol"]},
    {"resource_type": "sensor", "name_or_code": hydroweb_sensor_template["method_code"]},
    {"resource_type": "sensor", "name_or_code": geoglows_sensor_template["method_code"]},
    {"resource_type": "processing_level", "name_or_code": processing_level_template["code"]},
    {"resource_type": "result_qualifier", "name_or_code": result_qualifier_template["code"]},
])
display(metadata_templates)

## Create Needed Metadata and an Example Thing

In [ ]:
created = {}

if hs_api is None or workspace is None or workspace_uid is None:
    print("Skipping creation because an authenticated workspace is required.")
else:
    try:
        created["water_level_observed_property"] = record_resource("observed_property", hs_api.observedproperties.create(**water_level_observed_property_template))
        created["streamflow_observed_property"] = record_resource("observed_property", hs_api.observedproperties.create(**streamflow_observed_property_template))
        created["water_level_unit"] = record_resource("unit", hs_api.units.create(**water_level_unit_template))
        created["streamflow_unit"] = record_resource("unit", hs_api.units.create(**streamflow_unit_template))
        created["hydroweb_sensor"] = record_resource("sensor", hs_api.sensors.create(**hydroweb_sensor_template))
        created["geoglows_sensor"] = record_resource("sensor", hs_api.sensors.create(**geoglows_sensor_template))
        created["processing_level"] = record_resource("processing_level", hs_api.processinglevels.create(**processing_level_template))
        created["result_qualifier"] = record_resource("result_qualifier", hs_api.resultqualifiers.create(**result_qualifier_template))
        created["thing"] = record_resource("thing", hs_api.things.create(**build_thing_template(selected_station)), station_id=selected_station["ID"])
    except Exception as exc:
        print(f"Could not create metadata resources: {exc}")

display(created_resources_dataframe())

## Inspect Workspace Metadata

In [ ]:
if hs_api is None:
    print("Skipping live metadata inspection because the HydroServer client is unavailable.")
else:
    for label, endpoint_name in [
        ("things", "things"),
        ("datastreams", "datastreams"),
        ("observed properties", "observedproperties"),
        ("units", "units"),
        ("sensors", "sensors"),
    ]:
        try:
            endpoint = getattr(hs_api, endpoint_name)
            collection = endpoint.list(page_size=5, page=1)
            items = getattr(collection, "items", collection)
            print(f"{label}: showing up to {len(list(items)) if not isinstance(items, list) else len(items)} records from first page")
        except Exception as exc:
            print(f"Could not inspect {label}: {exc}")

## Cleanup: Delete Created Resources

In [ ]:
cleanup_order = [
    "task",
    "data_connection",
    "orchestration_system",
    "datastream",
    "thing",
    "result_qualifier",
    "processing_level",
    "sensor",
    "unit",
    "observed_property",
    "workspace",
]

preview = created_resources_dataframe()
if not preview.empty:
    display(preview)
else:
    print("No created resources are recorded.")

if not DELETE_CREATED_RESOURCES_AT_END:
    print("Cleanup skipped because DELETE_CREATED_RESOURCES_AT_END is False.")
elif hs_api is None:
    print("Cleanup skipped because the HydroServer client is unavailable.")
else:
    deleted_ids = set()
    for resource_type in cleanup_order:
        for row in reversed(created_resources):
            if row["resource_type"] != resource_type:
                continue
            resource = row["resource"]
            uid = resource_uid(resource)
            if resource is None or uid in deleted_ids:
                continue
            try:
                print(f"Deleting {resource_type}: {row['name_or_code']} | uuid={uid}")
                resource.delete()
                deleted_ids.add(uid)
            except Exception as exc:
                print(f"Could not delete {resource_type} {uid}: {exc}")
    print("Cleanup finished.")